# Stock Market Prediction using Random Forest Regression
**Technical indicators + current fundamental screening | Google Colab**

This notebook is adapted from the workflow in your Assignment_13.ipynb: data loading, feature engineering, train/test split, model fitting, evaluation, model saving and Excel export.

**Data:** Yahoo Finance through `yfinance` for adjusted historical OHLCV prices and current company information. This is a third-party research data route, not an official exchange feed. Verify important data with NSE/BSE filings and company annual reports.

**Prediction target:** next 20 trading-day return. This is an estimate, not a guaranteed future price.

## Step 1 — Install required libraries

In [ ]:
!pip -q install yfinance scikit-learn pandas numpy matplotlib openpyxl joblib

## Step 2 — Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Step 3 — Select a company and download its Excel-ready market data
Use NSE ticker suffix `.NS`. Examples: `RELIANCE.NS`, `TCS.NS`, `INFY.NS`, `ITC.NS`, `HDFCBANK.NS`.

In [ ]:
ticker = "RELIANCE.NS"   # Change this to any supported Yahoo Finance ticker
period = "5y"
price = yf.download(ticker, period=period, interval="1d", auto_adjust=True, progress=False)
if isinstance(price.columns, pd.MultiIndex):
    price.columns = price.columns.get_level_values(0)
price = price[["Open","High","Low","Close","Volume"]].dropna()
display(price.head())
print("Rows:", len(price), "| Start:", price.index.min(), "| End:", price.index.max())

## Step 4 — Download NIFTY 50 benchmark data

In [ ]:
benchmark = yf.download("^NSEI", period=period, interval="1d", auto_adjust=True, progress=False)
if isinstance(benchmark.columns, pd.MultiIndex):
    benchmark.columns = benchmark.columns.get_level_values(0)
benchmark_close = benchmark["Close"].reindex(price.index).ffill()

## Step 5 — Calculate technical indicators
RSI, MACD, moving averages, volatility, ATR, volume ratio and relative strength.

In [ ]:
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

data = price.copy()
close = data["Close"]
data["ret_1d"] = close.pct_change()
data["ret_5d"] = close.pct_change(5)
data["ret_20d"] = close.pct_change(20)
data["ret_60d"] = close.pct_change(60)
for n in [20,50,200]:
    data[f"sma{n}"] = close.rolling(n).mean()
    data[f"sma{n}_gap"] = close / data[f"sma{n}"] - 1
data["rsi14"] = calculate_rsi(close, 14)
ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()
data["macd"] = ema12 - ema26
data["macd_signal"] = data["macd"].ewm(span=9, adjust=False).mean()
data["macd_gap"] = data["macd"] / close
data["macd_signal_gap"] = (data["macd"] - data["macd_signal"]) / close
prev = close.shift(1)
tr = pd.concat([data["High"]-data["Low"], (data["High"]-prev).abs(), (data["Low"]-prev).abs()], axis=1).max(axis=1)
data["atr_pct"] = tr.rolling(14).mean() / close
data["volatility20"] = data["ret_1d"].rolling(20).std() * np.sqrt(252)
data["volume_ratio20"] = data["Volume"] / data["Volume"].rolling(20).mean()
data["relative_strength20"] = data["ret_20d"] - benchmark_close.pct_change(20)
display(data.tail())

## Step 6 — Create the prediction target
The target is the percentage return from today's close to the close 20 trading sessions ahead. The last 20 rows have no known target and are excluded from training.

In [ ]:
HORIZON = 20
data["future_return_20d"] = data["Close"].shift(-HORIZON) / data["Close"] - 1
features = [
    "ret_1d","ret_5d","ret_20d","ret_60d",
    "sma20_gap","sma50_gap","sma200_gap",
    "rsi14","macd_gap","macd_signal_gap",
    "volatility20","atr_pct","volume_ratio20","relative_strength20"
]
model_data = data.dropna(subset=features + ["future_return_20d"]).copy()
X = model_data[features]
y = model_data["future_return_20d"]
print("X shape:", X.shape, "| y shape:", y.shape)

## Step 7 — Chronological train/test split
Unlike the random split in the original classification assignment, time-series data must be split chronologically. The latest 20% is held out to test future-period performance.

In [ ]:
split = int(len(model_data) * 0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
print("Training:", X_train.shape, "Testing:", X_test.shape)

## Step 8 — Train the Random Forest Regression model

In [ ]:
model = RandomForestRegressor(
    n_estimators=400, max_depth=8, min_samples_leaf=5,
    max_features=0.8, random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

## Step 9 — Evaluate model performance
MAE and RMSE measure average prediction error; R² measures fit relative to the test-set mean. A negative R² means the model performed worse than predicting the test-set average.

In [ ]:
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)
metrics = pd.DataFrame({"Metric":["MAE","RMSE","R2"],"Value":[mae,rmse,r2]})
display(metrics)

## Step 10 — Refit on all labelled observations and predict the latest 20-day return

In [ ]:
final_model = RandomForestRegressor(
    n_estimators=400, max_depth=8, min_samples_leaf=5,
    max_features=0.8, random_state=42, n_jobs=-1
)
final_model.fit(X, y)
latest_row = data.dropna(subset=features).iloc[[-1]]
predicted_return = float(final_model.predict(latest_row[features])[0])
latest_price = float(price["Close"].iloc[-1])
estimated_price = latest_price * (1 + predicted_return)
print("Latest close:", round(latest_price,2))
print("Estimated 20-trading-day return:", f"{predicted_return:.2%}")
print("Illustrative price estimate:", round(estimated_price,2))

## Step 11 — Fundamental analysis
This retrieves the current company snapshot where Yahoo Finance provides it. These are not historical point-in-time fundamentals, so they are not included in the historical regression features.

In [ ]:
info = yf.Ticker(ticker).get_info()
fundamentals = {
    "Company": info.get("longName", ticker),
    "Sector": info.get("sector"),
    "Trailing P/E": info.get("trailingPE"),
    "Forward P/E": info.get("forwardPE"),
    "Price to Book": info.get("priceToBook"),
    "ROE": info.get("returnOnEquity"),
    "ROA": info.get("returnOnAssets"),
    "Debt to Equity": info.get("debtToEquity"),
    "Profit Margin": info.get("profitMargins"),
    "Revenue Growth": info.get("revenueGrowth"),
    "Earnings Growth": info.get("earningsGrowth"),
    "Dividend Yield": info.get("dividendYield")
}
display(pd.DataFrame({"Metric": fundamentals.keys(), "Value": fundamentals.values()}))

## Step 12 — Calculate a transparent fundamental screening score
This broad score is illustrative, not sector-adjusted. A bank's leverage and valuation should not be compared mechanically with an IT company.

In [ ]:
checks = [
    ("ROE", fundamentals["ROE"], lambda v: v >= 0.15, 20),
    ("Profit Margin", fundamentals["Profit Margin"], lambda v: v >= 0.10, 20),
    ("Revenue Growth", fundamentals["Revenue Growth"], lambda v: v > 0, 15),
    ("Earnings Growth", fundamentals["Earnings Growth"], lambda v: v > 0, 15),
    ("Trailing P/E", fundamentals["Trailing P/E"], lambda v: 0 < v < 30, 15),
    ("Price to Book", fundamentals["Price to Book"], lambda v: 0 < v < 5, 10),
    ("Debt to Equity", fundamentals["Debt to Equity"], lambda v: v < 100, 10)
]
earned = available = 0
screen_rows = []
for name, value, rule, weight in checks:
    if value is None or not np.isfinite(value):
        result = "Not available"
    else:
        passed = rule(value)
        earned += weight if passed else 0
        available += weight
        result = "Pass" if passed else "Review"
    screen_rows.append([name, value, weight, result])
fundamental_score = earned / available * 100 if available else np.nan
display(pd.DataFrame(screen_rows, columns=["Metric","Value","Weight","Screening result"]))
print("Fundamental screening score:", "N/A" if pd.isna(fundamental_score) else f"{fundamental_score:.1f}/100")

## Step 13 — Combine technical prediction and fundamental screen

In [ ]:
if predicted_return >= 0.05 and (pd.isna(fundamental_score) or fundamental_score >= 60):
    signal = "BUY CANDIDATE"
elif predicted_return <= -0.03:
    signal = "AVOID / WEAK"
else:
    signal = "HOLD / WATCH"
summary = pd.DataFrame([{
    "Ticker": ticker, "Company": fundamentals["Company"],
    "Latest Price": latest_price, "Predicted 20D Return": predicted_return,
    "Illustrative Estimated Price": estimated_price,
    "Fundamental Score": fundamental_score, "Screening Signal": signal,
    "MAE": mae, "RMSE": rmse, "R2": r2
}])
display(summary)

## Step 14 — Visualise price and model feature importance

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(price.index, price["Close"])
plt.title(f"{ticker} adjusted closing price")
plt.xlabel("Date"); plt.ylabel("Price"); plt.grid(True); plt.show()

importance = pd.Series(final_model.feature_importances_, index=features).sort_values()
importance.plot(kind="barh", figsize=(8,5), title="Random Forest feature importance")
plt.show()

## Step 15 — Export the dataset and analysis into Excel

In [ ]:
excel_file = "stock_market_analysis.xlsx"
with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    price.to_excel(writer, sheet_name="Raw_Prices")
    data.to_excel(writer, sheet_name="Technical_Features")
    summary.to_excel(writer, sheet_name="Latest_Analysis", index=False)
    pd.DataFrame([fundamentals]).drop(columns=[], errors="ignore").to_excel(writer, sheet_name="Fundamentals", index=False)
    metrics.to_excel(writer, sheet_name="Model_Metrics", index=False)
    importance.rename("Importance").to_frame().to_excel(writer, sheet_name="Feature_Importance")
print("Created:", excel_file)

In [ ]:
from google.colab import files
files.download("stock_market_analysis.xlsx")

## Step 16 — Save the model for later use

In [ ]:
joblib.dump(final_model, "random_forest_stock_model.joblib")
files.download("random_forest_stock_model.joblib")

## Interpretation and limitations
- **MAE/RMSE:** lower is better, but compare against a simple baseline before trusting the model.
- **R²:** can be negative for financial return forecasts; this is not necessarily a coding error.
- **Fundamental score:** broad screening only; use sector peers, annual reports, cash flows, management quality and valuation context.
- **Buy candidate / Hold / Avoid:** a simple research label, not a personalised recommendation.
- A rigorous investment backtest must account for transaction costs, slippage, dividends, corporate actions, survivorship bias and multiple test periods.